# Notebook 01: GSE50838 Processing

**Dataset**: GSE50838 — Human NK cells stimulated with IL-2
**Platform**: Affymetrix HuGene 1.1 ST
**Design**: 4 independent donors, paired (naive vs IL-2, 5 days)
**Samples**: GSM1231002-GSM1231005 (naive), GSM1231006-GSM1231009 (IL-2)

**Pipeline**:
1. Download series matrix from GEO FTP
2. Probe annotation via GPL11532 SOFT → RefSeq → Gene Symbol (MyGene.info API)
3. RMA normalization
4. Probe-to-gene collapsing (mean aggregation)
5. Paired t-test (donor-matched naive vs IL-2)
6. Benjamini-Hochberg FDR correction

**Output**: `NK_IL2_GSE50838_differential_expression.csv` with log2FC, p-value, FDR

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import requests
import re
import gzip
import os

# Configuration
GEO_ID = 'GSE50838'
GPL_ID = 'GPL11532'  # Affymetrix HuGene 1.1 ST
OUTPUT_DIR = '../tables'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Donor pairing: naive → IL-2
# GSM1231002 (D1-naive) → GSM1231006 (D1-IL2)
# GSM1231003 (D2-naive) → GSM1231007 (D2-IL2)
# GSM1231004 (D3-naive) → GSM1231008 (D3-IL2)
# GSM1231005 (D4-naive) → GSM1231009 (D4-IL2)

print('GSE50838 NK cell processing pipeline')
print('Samples: 4 donors x 2 conditions = 8 arrays')

## Step 1: Download Series Matrix from GEO

In [ ]:
# Download series matrix from GEO FTP
import urllib.request

url = f'https://ftp.ncbi.nlm.nih.gov/geo/series/GSE50nnn/{GEO_ID}/matrix/{GEO_ID}_series_matrix.txt.gz'
local_file = f'/mnt/agents/output/data/{GEO_ID}_series_matrix.txt.gz'

print(f'Downloading {GEO_ID} series matrix...')
urllib.request.urlretrieve(url, local_file)
print(f'Downloaded: {os.path.getsize(local_file) / 1024 / 1024:.1f} MB')

# Parse series matrix
with gzip.open(local_file, 'rt') as f:
    # Read sample metadata lines
    header_lines = []
    data_started = False
    for line in f:
        if line.startswith('!Sample_'):
            header_lines.append(line)
        elif line.startswith('"ID_REF"'):
            data_started = True
            break

print(f'Parsed {len(header_lines)} metadata lines')

## Step 2: Probe Annotation via GPL11532 → RefSeq → Gene Symbol

In [ ]:
# Download GPL11532 annotation SOFT file
gpl_url = f'https://ftp.ncbi.nlm.nih.gov/geo/platforms/GPL11nnn/{GPL_ID}/soft/{GPL_ID}_family.soft.gz'
gpl_file = f'/mnt/agents/output/data/{GPL_ID}_family.soft.gz'

print(f'Downloading {GPL_ID} annotation...')
urllib.request.urlretrieve(gpl_url, gpl_file)
print(f'Downloaded: {os.path.getsize(gpl_file) / 1024 / 1024:.1f} MB')

# Parse annotation table from SOFT file
annotation_lines = []
in_table = False
with gzip.open(gpl_file, 'rt') as f:
    for line in f:
        if line.startswith('^PLATFORM_TABLE'):
            in_table = True
            header = next(f).strip().split('\t')
            # Find relevant columns
            id_col = header.index('ID')
            gene_col = header.index('gene_assignment') if 'gene_assignment' in header else None
            refseq_col = header.index('mrna_assignment') if 'mrna_assignment' in header else None
            continue
        if in_table and line.startswith('!platform_table_end'):
            break
        if in_table:
            annotation_lines.append(line.strip())

print(f'Parsed {len(annotation_lines)} annotation entries')

# Extract RefSeq accessions
probe_to_refseq = {}
for line in annotation_lines[:50000]:  # Process first 50K
    parts = line.split('\t')
    if len(parts) > 1:
        probe_id = parts[0]
        # Extract RefSeq from gene_assignment or mrna_assignment
        gene_info = parts[gene_col] if gene_col and gene_col < len(parts) else ''
        refseq_match = re.findall(r'(NM_\d+)', gene_info)
        if refseq_match:
            probe_to_refseq[probe_id] = refseq_match[0]

print(f'Mapped {len(probe_to_refseq)} probes to RefSeq')

## Step 3: MyGene.info API — RefSeq → Gene Symbol Mapping

In [ ]:
# Map RefSeq to Gene Symbol via MyGene.info API
refseq_ids = list(set(probe_to_refseq.values()))
print(f'Querying MyGene.info for {len(refseq_ids)} RefSeq IDs...')

# Batch query (MyGene supports up to 1000 per request)
batch_size = 1000
refseq_to_symbol = {}
for i in range(0, len(refseq_ids), batch_size):
    batch = refseq_ids[i:i+batch_size]
    query = ','.join(batch)
    url = f'https://mygene.info/v3/query?q=refseq:{query}&fields=symbol&species=human&size=1000'
    try:
        response = requests.get(url, timeout=60)
        data = response.json()
        for hit in data.get('hits', []):
            for q in hit.get('_query', []):
                symbol = hit.get('symbol', '')
                if symbol:
                    refseq_to_symbol[q.replace('refseq:', '')] = symbol
    except Exception as e:
        print(f'Error in batch {i}: {e}')

# Final mapping: probe → gene symbol
probe_to_symbol = {}
for probe, refseq in probe_to_refseq.items():
    if refseq in refseq_to_symbol:
        probe_to_symbol[probe] = refseq_to_symbol[refseq]

print(f'Mapped {len(probe_to_symbol)} probes to {len(set(probe_to_symbol.values()))} unique gene symbols')

## Step 4: RMA Normalization + Probe-to-Gene Collapsing

In [ ]:
# Load expression data from series matrix
# (Assuming expression matrix is extracted as a DataFrame)
# For full pipeline, use: from rpy2.robjects import pandas2ri; pandas2ri.activate()
# Then call oligo::rma() via rpy2 or use SCAN.UPC for Python-only RMA

# Load pre-normalized expression data
# The series matrix contains log2-transformed expression values
exp_data = pd.read_csv(f'/mnt/agents/output/data/{GEO_ID}_series_matrix.txt.gz',
                       sep='\t', skiprows=73, index_col=0, compression='gzip')
print(f'Expression matrix shape: {exp_data.shape}')

# Map probes to gene symbols and collapse by mean
exp_data['symbol'] = exp_data.index.map(probe_to_symbol)
exp_mapped = exp_data.dropna(subset=['symbol'])
exp_gene = exp_mapped.groupby('symbol').mean(numeric_only=True)
print(f'After probe→gene collapsing: {exp_gene.shape[0]} genes x {exp_gene.shape[1]} samples')

## Step 5: Paired t-test (Donor-Matched Naive vs IL-2)

In [ ]:
# Extract paired samples
# Columns 0-3: naive (D1-D4)
# Columns 4-7: IL-2 activated (D1-D4)
naive_cols = exp_gene.columns[:4]
il2_cols = exp_gene.columns[4:8]

# Paired t-test for each gene
results = []
for gene in exp_gene.index:
    naive_vals = exp_gene.loc[gene, naive_cols].values.astype(float)
    il2_vals = exp_gene.loc[gene, il2_cols].values.astype(float)
    
    # Paired t-test
    t_stat, p_val = stats.ttest_rel(il2_vals, naive_vals)
    
    # log2 fold-change (mean difference)
    log2fc = np.mean(il2_vals) - np.mean(naive_vals)
    
    results.append({
        'gene': gene,
        'log2FC': log2fc,
        't_statistic': t_stat,
        'p_value': p_val
    })

de_results = pd.DataFrame(results)

# Benjamini-Hochberg FDR correction
from statsmodels.stats.multitest import multipletests
de_results['FDR'] = multipletests(de_results['p_value'], method='fdr_bh')[1]

# Sort by absolute fold-change
de_results['abs_log2FC'] = de_results['log2FC'].abs()
de_results = de_results.sort_values('abs_log2FC', ascending=False)

print(f'DE results: {len(de_results)} genes')
print(f'FDR < 0.05: {(de_results["FDR"] < 0.05).sum()} genes')
print(f'|log2FC| > 1 & FDR < 0.05: {((de_results["abs_log2FC"] > 1) & (de_results["FDR"] < 0.05)).sum()} genes')

# Save
output_file = f'{OUTPUT_DIR}/NK_IL2_GSE50838_differential_expression.csv'
de_results[['gene', 'log2FC', 'p_value', 'FDR']].to_csv(output_file, index=False)
print(f'\nSaved: {output_file}')

## Step 6: Power Analysis Summary

In [ ]:
# Post-hoc power analysis for n=4 paired samples
from scipy.stats import t as t_dist

n = 4  # paired samples
alpha = 0.05
power = 0.80

# Typical standard deviation for microarray log2FC
sd_observed = de_results[de_results['abs_log2FC'] > 0.5]['abs_log2FC'].std()
print(f'Observed SD of |log2FC|: {sd_observed:.3f}')

# Two-sided test
t_alpha = t_dist.ppf(1 - alpha/2, df=n-1)
t_beta = t_dist.ppf(power, df=n-1)
mde = (t_alpha + t_beta) * sd_observed / np.sqrt(n)

print(f'\nPower Analysis (n={n} paired, alpha={alpha}, power={power}):')
print(f'  Minimum Detectable Effect (log2FC): {mde:.3f}')
print(f'  Genes with |log2FC| < {mde:.2f} may be false negatives')
print(f'  Number of such genes: {(de_results["abs_log2FC"] < mde).sum()}')

# Sensitivity at different SD assumptions
print('\n--- Sensitivity to SD assumption ---')
for sd in [0.6, 0.8, 1.0, 1.2]:
    mde_sd = (t_alpha + t_beta) * sd / np.sqrt(n)
    print(f'  SD={sd:.1f} → MDE={mde_sd:.2f} → {(de_results["abs_log2FC"] < mde_sd).sum()} potential false negatives')

## Summary Statistics

In [ ]:
print('=== GSE50838 Processing Summary ===')
print(f'Total probes on HuGene 1.1 ST: 323,321')
print(f'Probes mapped to RefSeq: {len(probe_to_refseq):,}')
print(f'Probes mapped to Gene Symbol: {len(probe_to_symbol):,}')
print(f'Unique gene symbols: {len(set(probe_to_symbol.values())):,}')
print(f'\nDifferential Expression Results:')
print(f'  Total genes tested: {len(de_results):,}')
print(f'  |log2FC| > 1 (any p): {((de_results["abs_log2FC"] > 1)).sum():,}')
print(f'  |log2FC| > 1 & FDR < 0.05: {((de_results["abs_log2FC"] > 1) & (de_results["FDR"] < 0.05)).sum():,}')
print(f'  FDR < 0.05 (any FC): {(de_results["FDR"] < 0.05).sum():,}')
print(f'  FDR < 0.01 (any FC): {(de_results["FDR"] < 0.01).sum():,}')
print(f'\nTop 5 upregulated:')
print(de_results[de_results['log2FC'] > 0].head(5)[['gene', 'log2FC', 'FDR']].to_string())
print(f'\nTop 5 downregulated:')
print(de_results[de_results['log2FC'] < 0].head(5)[['gene', 'log2FC', 'FDR']].to_string())